<a href="https://colab.research.google.com/github/nashsparrow/pytorch/blob/main/2.pytorch_cifar10_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

import numpy as np
import pandas as pd

Plan,
Import Dataset
Split the train and Test data
Check the shape
Batch Them

create the model
CV 1
pool 1
CV2
pool 2
CV3
pool 3
FC 1
FC 2
FC 3 - out

create the forward function

Instantiate the model
define criteria - loss function
define optimizer - adam optimizer

define epochs
loop epochs
loop batch

train
and test




In [ ]:
#Dataset loading, Transform
transform = transforms.ToTensor() #change from Height, Width, Channels to C, H W

train_data = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

In [ ]:
#loaders
train_loader = torch.utils.data.DataLoader(train_data, batch_size=10, shuffle=True) #processed by batches, to memory enhancements #simultaniously process by gpu inside a batch
test_loader = torch.utils.data.DataLoader(test_data, batch_size=10, shuffle=True)

In [ ]:
#Model
class ConvolutionalNeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    #input 3 x 32 x 32
    self.conv1 = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3) #output 8 x 30 x 30
    self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) #output 8 x 15 x 15
    self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1) #output 16 x 15 x 15
    self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) #output 16 x 7 x 7
    self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3) #output 32 x 5 x 5
    self.fc1 = nn.Linear(in_features=32*5*5, out_features=1024)
    self.fc2 = nn.Linear(in_features=1024, out_features=512)
    self.fc3 = nn.Linear(in_features=512, out_features=10)

  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool1(x)
    x = F.relu(self.conv2(x))
    x = self.pool2(x)
    x = F.relu(self.conv3(x))
    x = x.view(-1, 32*5*5) #Batch size can be vary
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return F.log_softmax(x, dim=1)

In [ ]:
#create the model
model = ConvolutionalNeuralNetwork()
model

ConvolutionalNeuralNetwork(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=800, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=10, bias=True)
)

In [ ]:
#Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 5 #training cycles
for epoch in range(epochs):
  for i, (x_train, y_train) in enumerate(train_loader): #Enumerate gives you counter
    y_pred = model(x_train) #Pass through the model and get prediction for the batch
    loss = criterion(y_pred, y_train) #get the loss of the batch

    #update parameters
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i%600 == 0:
      print(f'epoch: {epoch} batch: {i} loss: {loss.item()}')

  with torch.no_grad(): #no gradient
    for i, (x_test, y_test) in enumerate(test_loader):
      y_pred = model(x_test)
      loss = criterion(y_pred, y_test)

epoch: 0 batch: 0 loss: 1.6653531789779663
epoch: 0 batch: 600 loss: 1.9936469793319702
epoch: 0 batch: 1200 loss: 0.841163158416748
epoch: 0 batch: 1800 loss: 2.0236501693725586
epoch: 0 batch: 2400 loss: 0.8962246179580688
epoch: 0 batch: 3000 loss: 0.9858708381652832
epoch: 0 batch: 3600 loss: 1.4869073629379272
epoch: 0 batch: 4200 loss: 1.255501389503479
epoch: 0 batch: 4800 loss: 1.1463449001312256
epoch: 1 batch: 0 loss: 1.2643495798110962
epoch: 1 batch: 600 loss: 0.6874302625656128
epoch: 1 batch: 1200 loss: 1.4031704664230347
epoch: 1 batch: 1800 loss: 1.2678368091583252
epoch: 1 batch: 2400 loss: 1.157199501991272
epoch: 1 batch: 3000 loss: 1.1633217334747314
epoch: 1 batch: 3600 loss: 1.1242609024047852
epoch: 1 batch: 4200 loss: 1.238654375076294
epoch: 1 batch: 4800 loss: 1.3410742282867432
epoch: 2 batch: 0 loss: 0.9089207649230957
epoch: 2 batch: 600 loss: 0.6884652376174927
epoch: 2 batch: 1200 loss: 0.39128318428993225
epoch: 2 batch: 1800 loss: 1.7491772174835205
epo